Скачивание ruFacts-fact-checking датасет (русскоязычный)
https://huggingface.co/datasets/akozlova/RuFacts

In [60]:
import json
import pandas as pd
import random

random.seed(42)


data_train = []
with open("../data/raw/train.json", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            ex = json.loads(line)
            data_train.append({
                "evidence": ex["evidence"],
                "claim": ex["claim"],
                "label": ex["label"]
            })
df_train = pd.DataFrame(data_train)

data_val = []
with open("../data/raw/validation.json", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            ex = json.loads(line)
            data_val.append({
                "evidence": ex["evidence"],
                "claim": ex["claim"],
                "label": ex["label"]
            })
df_val = pd.DataFrame(data_val)

df = pd.concat([df_train, df_val], ignore_index=True)

df = df.rename(columns={
    "evidence": "prompt",
    "claim": "model_answer",
    "label": "is_hallucination"
})
df["correct_answer"] = None
df["comment"] = None

print(f"Examples: {len(df)}")
print(f"True: {len(df[df['is_hallucination']==0])}")

Examples: 6236
True: 2994


In [62]:
df_true = df[df["is_hallucination"] == 0].copy().reset_index(drop=True)

In [63]:
df_true.sample(3)

,prompt,model_answer,is_hallucination,correct_answer,comment
2964,"В Индии, на втором по объему развивающемся авт...",В Индии на втором по объему развивающемся рынк...,0,None,None
141,Их сложные крепления явно приходились ближайше...,Их диковинные крепления выглядели как двоюродн...,0,None,None
1553,Разница состоит лишь в сравнительно более позд...,В целом разница между этими двумя политиками в...,0,None,None


добавляю мутации

In [64]:
import re
import random

class FactualMutator:
    def __init__(self, seed=42):
        self.rng = random.Random(seed)
        
        self.units = {
          
            "руб.": "долл.", "долл.": "евро", "евро": "юань", "юань": "руб.",
            "рублей": "долларов", "долларов": "евро", "евро": "фунтов",
           
            "млн": "тыс.", "тыс.": "млн", "млрд": "млн", "млн": "млрд",
            "процент": "пункт", "пункт": "процент", "%": "промилле",
    
            "км": "м", "м": "км", "см": "мм", "мм": "см",
            "миль": "км", "км": "миль",
           
            "кг": "т", "т": "кг", "г": "кг", "кг": "г",
            "литр": "мл", "мл": "литр", "куб. м": "куб. км",
           
            "лет": "месяцев", "месяцев": "недель", "дней": "часов"
        }
        
        self.negations = {
          
            "не ": "",
            "не был": "был", "не была": "была", "не было": "было", "не были": "были",
            "отсутствует": "присутствует", "присутствует": "отсутствует",
            "нет ": "есть ", "есть ": "нет ",
            "ложно": "верно", "верно": "ложно",
            "всегда": "никогда", "никогда": "всегда",
            "часто": "редко", "редко": "часто",
            "запрещено": "разрешено", "разрешено": "запрещено"
        }
        
        self.names = {
            
            "Путин": "Медведев", "Медведев": "Путин",
            "Сталин": "Ленин", "Ленин": "Сталин", "Троцкий": "Сталин",
            "Петр I": "Николай II", "Николай II": "Петр I", "Екатерина II": "Елизавета",
            "Хрущев": "Брежнев", "Брежнев": "Андропов", "Горбачев": "Ельцин", "Ельцин": "Путин",
           
            "Байден": "Трамп", "Трамп": "Байден", "Обама": "Буш",
            "Макрон": "Шольц", "Шольц": "Макрон", "Джонсон": "Сунак",
            "Си Цзиньпин": "Ли Кэцян", "Ким Чен Ын": "Ким Чен Ир",
       
            "Пушкин": "Лермонтов", "Лермонтов": "Гоголь", "Толстой": "Достоевский", 
            "Достоевский": "Тургенев", "Чехов": "Бунин",
            "Есенин": "Маяковский", "Маяковский": "Блок",
            "Бетховен": "Моцарт", "Моцарт": "Бах", "Шекспир": "Диккенс"
        }
        
        self.geo =  {
           
            "Россия": "СССР", "СССР": "Россия", "РСФСР": "СССР",
            "США": "Канада", "Канада": "Мексика", "Мексика": "Бразилия",
            "Германия": "Франция", "Франция": "Италия", "Италия": "Испания", "Испания": "Португалия",
            "Великобритания": "Англия", "Англия": "Шотландия", "Шотландия": "Уэльс",
            "Китай": "Япония", "Япония": "Корея", "Корея": "Вьетнам",
            "Украина": "Беларусь", "Беларусь": "Казахстан", "Казахстан": "Узбекистан",
            
            "Москва": "Санкт-Петербург", "Санкт-Петербург": "Новосибирск",
            "Лондон": "Париж", "Париж": "Берлин", "Берлин": "Рим", "Рим": "Мадрид",
            "Нью-Йорк": "Лос-Анджелес", "Лос-Анджелес": "Чикаго",
            "Пекин": "Шанхай", "Шанхай": "Токио", "Токио": "Сеул",
            "Киев": "Минск", "Минск": "Астана", "Астана": "Ташкент"
        }
        
        self.dates = {
            "января": "февраля", "февраля": "марта", "марта": "апреля",
            "понедельник": "вторник", "вторник": "среду", "среду": "четверг",
            "утром": "вечером", "вечером": "ночью", "ночью": "утром",
            "весной": "летом", "летом": "осенью", "осенью": "зимой"
        }


        self.quantifiers = {
            "один": "два", "два": "три", "три": "пять", "пять": "десять",
            "первый": "второй", "второй": "третий", "третий": "четвертый",
            "половина": "треть", "треть": "четверть", "четверть": "половина",
            "несколько": "множество", "множество": "большинство", "большинство": "меньшинство"
        }

    def _mutate_year(self, text):
        def change_year(m):
            return str(int(m.group(0)) + self.rng.choice([-7, -3, 2, 5, 12]))
        return re.sub(r'\b(1[0-9]{3}|20[0-2][0-9])\b', change_year, text, count=1)
    
    def _mutate_number(self, text):
        def change_int(m):
            return str(int(int(m.group(0)) * self.rng.choice([0.5, 1.5, 2.0])))
        return re.sub(r'\b\d{3,6}\b', change_int, text, count=1)
        
    def _mutate_decimal(self, text):
        def change_float(m):
            val = float(m.group(0).replace(',', '.'))
            return str(round(val * self.rng.choice([1.3, 0.7, 2.1]), 1))
        return re.sub(r'\b\d+[.,]\d+\b', change_float, text, count=1)

    def _safe_replace_dict(self, text, replace_dict):
        
        for old, new in replace_dict.items():
            if ' ' in old or not old.strip().isalnum():
                if old in text:
                    return text.replace(old, new, 1)
        

        for old, new in replace_dict.items():
            if ' ' in old or not old.strip().isalnum():
                continue 
            pattern = r'\b' + re.escape(old.strip()) + r'\b'
            if re.search(pattern, text):
                return re.sub(pattern, new, text, count=1)
        return text

    def mutate(self, text):
        if pd.isna(text):
            return str(text), False
            
        original = text
        mutations = []
        
        if re.search(r'\b(1[0-9]{3}|20[0-2][0-9])\b', text):
            mutations.append(('year', self._mutate_year))
        if re.search(r'\b\d{3,6}\b', text):
            mutations.append(('int', self._mutate_number))
        if re.search(r'\b\d+[.,]\d+\b', text):
            mutations.append(('dec', self._mutate_decimal))
        if any(unit in text for unit in self.units):
            mutations.append(('unit', lambda t: self._safe_replace_dict(t, self.units)))
        if any(name in text for name in self.names):
            mutations.append(('name', lambda t: self._safe_replace_dict(t, self.names)))
        if any(place in text for place in self.geo):
            mutations.append(('geo', lambda t: self._safe_replace_dict(t, self.geo)))
        if any(word in text for word in self.negations):
            mutations.append(('neg', lambda t: self._safe_replace_dict(t, self.negations)))
        if any(word in text for word in self.dates):
            mutations.append(('date', lambda t: self._safe_replace_dict(t, self.dates)))
        if any(word in text for word in self.quantifiers):
            mutations.append(('quant', lambda t: self._safe_replace_dict(t, self.quantifiers)))
            
        if not mutations:
            return text, False, None
            
        mutation_type, mutate_func = self.rng.choice(mutations)
        mutated = mutate_func(text)
        
        if mutated != original:
            return mutated, True, mutation_type  
        return text, False, None

In [65]:
from tqdm.auto import tqdm


mutator = FactualMutator(seed=42)

rows = []
fail = 0
mutation_st = {}

for _, row in tqdm(df_true.iterrows(), total=len(df_true), desc="mutation"):
    mutated_text, success, m_type = mutator.mutate(row["model_answer"])
    
    # success уже гарантирует, что mutated_text != original, но двойная проверка безопасна
    if success:
        if m_type in mutation_st:
            mutation_st[m_type] += 1
        else:
            mutation_st[m_type] = 1
            
        rows.append({
            "prompt": row["prompt"],
            "model_answer": row["model_answer"],
            "is_hallucination": 0,
            "correct_answer": None,
            "comment": None
        })
        rows.append({
            "prompt": row["prompt"],
            "model_answer": mutated_text,
            "is_hallucination": 1,
            "correct_answer": None,
            "comment": "synthetic_fact_mutation"
        })
    else:
        fail += 1

df_train = pd.DataFrame(rows)
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

mask_pairs = df_train.groupby("prompt")["is_hallucination"].transform("nunique") == 2
df_train = df_train[mask_pairs].reset_index(drop=True)

print(f" {df_train['is_hallucination'].value_counts(normalize=True).to_dict()}")
print(f"fail_generation = {fail}")


SAVE_PATH = "../data/train.csv"  
df_train.to_csv(SAVE_PATH, index=False)


mutation:   0%|          | 0/2994 [00:00<?, ?it/s]

 {0: 0.5, 1: 0.5}
fail_generation = 2105


In [66]:
total_mutated = sum(mutation_st.values())
if total_mutated > 0:
    for m_type in sorted(mutation_st, key=mutation_st.get, reverse=True):
        count = mutation_st[m_type]
        pct = count / total_mutated * 100
        print(f"  {m_type:8s}: {count:4d} ({pct:5.1f}%)")

  neg     :  353 ( 39.7%)
  unit    :  149 ( 16.8%)
  quant   :  101 ( 11.4%)
  geo     :   96 ( 10.8%)
  int     :   88 (  9.9%)
  year    :   45 (  5.1%)
  name    :   25 (  2.8%)
  date    :   17 (  1.9%)
  dec     :   15 (  1.7%)


In [67]:
sample_prompt = df_train.groupby("prompt").filter(lambda x: x["is_hallucination"].nunique() == 2).iloc[0]["prompt"]
pair = df_train[df_train["prompt"] == sample_prompt]

print("prompt:", pair["prompt"].iloc[0])
print("correct: ", pair[pair["is_hallucination"]==0]["model_answer"].iloc[0])
print("incorrect: ", pair[pair["is_hallucination"]==1]["model_answer"].iloc[0])


prompt: А вы думали Илон Рив Маск сам всякие штуки придумывает, и за ним никто не стоит? Ага, щас. Просто коты из-за своей природной скромности предпочитают в тени оставаться.
correct:  А ты думаешь, что Илон Рив Маск сам все придумывает, и за ним никто не стоит? Ага, щас. Просто кошки из-за своей природной скромности предпочитают оставаться в тени.
incorrect:  А ты думаешь, что Илон Рив Маск сам все придумывает, и за ним никто стоит? Ага, щас. Просто кошки из-за своей природной скромности предпочитают оставаться в тени.


In [68]:
df_train.sample(10)

,prompt,model_answer,is_hallucination,correct_answer,comment
405,В Красноярском крае при пожаре погибли 6 детей.,При пожаре в Красноярском крае погибли шесть д...,0,None,None
1499,Цены на нефть падают в связи с информацией о р...,Стоимость нефти снижается на данных о росте ее...,0,None,None
160,"Трамп: ""Я опустил цены на нефть, и С.Аравия по...","Трамп: « Я сбросил цены на нефть, и С. Аравия ...",0,None,None
1634,"Анфиса Резцова: ""Сборная России по биатлону де...","Анфиса Резцова: У одного ходьбы не хватает, у ...",0,None,None
700,"Доктор Персон задержался у стойки, чтобы освед...","Доктор Персон приостановился у стойки портье, ...",1,None,synthetic_fact_mutation
210,Клеопатра (1963 фильм) рассказывает о борьбе м...,Клеопатра - это американский эпический историч...,1,None,synthetic_fact_mutation
1364,"В дальнейшем, с развитием маржиналистских теор...",Затем в результате разработки маргиналистов пр...,1,None,synthetic_fact_mutation
170,"Благодаря развитию коммерческих центров, откры...","Благодаря развитию коммерческих центров, откры...",1,None,synthetic_fact_mutation
916,Пентагон разрабатывает новую стратегию военных...,Генштаб США разработал новую стратегию военных...,0,None,None
91,"Голосование прошло 20 марта 2011г., передает Б...","Голосование прошло 20 апреля 2011г., передает ...",1,None,synthetic_fact_mutation


In [69]:
df_train.shape

(1778, 5)